# 👗 StyleMate AI — Open-Source Model Fine-Tuning (LoRA / SFT)

Welcome to the **StyleMate Fine-Tuning Pipeline**! This turnkey Google Colab notebook fine-tunes an open-source lightweight LLM (**Qwen2.5-1.5B-Instruct** or **Llama-3.2-1B-Instruct**) to act as the specialized Natural Language Understanding (NLU) and stylist intelligence engine for StyleMate.

### 🎯 Objectives:
1. Extract structured fashion criteria (intent, occasion, season, style, color, time of day) directly to JSON.
2. Generate personalized stylist advice and tone.
3. Run in **~3-5 minutes on a free Google Colab T4 GPU**.
4. Export the resulting LoRA adapter for local offline inference in StyleMate without cloud API costs.

---  
## ⚙️ Step 1: Install Fast Fine-Tuning Libraries (1-Click Setup)
We use Hugging Face `transformers`, `peft`, `trl`, `datasets`, and `accelerate` (or `unsloth` for 2x faster execution).

In [ ]:
# Install Hugging Face PEFT, TRL, and Accelerate
!pip install -q --upgrade pip
!pip install -q "torch>=2.1.0" "transformers>=4.44.0" "peft>=0.12.0" "trl>=0.9.6" "datasets>=2.20.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.0"

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

---  
## 📦 Step 2: Load StyleMate `train_nlu.jsonl` Dataset
We load the curated fashion dataset containing multi-turn styling interactions, domain-specific edge cases, and JSON schema targets.

In [ ]:
import json
from datasets import Dataset

# Ground truth dataset from StyleMate
sample_data = [
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "Need a sharp outfit for an executive client presentation tomorrow morning"}, {"role": "assistant", "content": json.dumps({"intent":"outfit_recommendation","season":None,"occasion":"office","time_of_day":"morning","weather_note":None,"preferred_color":None,"preferred_style":"formal","conversational_response":None})}]},
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "What should I wear to a final round job interview at a tech company?"}, {"role": "assistant", "content": json.dumps({"intent":"outfit_recommendation","season":None,"occasion":"interview","time_of_day":None,"weather_note":None,"preferred_color":None,"preferred_style":"smart","conversational_response":None})}]},
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "Casual Friday outfit for the office with navy chinos"}, {"role": "assistant", "content": json.dumps({"intent":"outfit_recommendation","season":None,"occasion":"office","time_of_day":None,"weather_note":None,"preferred_color":"navy","preferred_style":"smart","conversational_response":None})}]},
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "Going to a wedding reception this Saturday evening, need something elegant"}, {"role": "assistant", "content": json.dumps({"intent":"outfit_recommendation","season":None,"occasion":"wedding","time_of_day":"evening","weather_note":None,"preferred_color":None,"preferred_style":"formal","conversational_response":None})}]},
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "First date at an intimate Italian restaurant tonight, want to look stylish but effortless"}, {"role": "assistant", "content": json.dumps({"intent":"outfit_recommendation","season":None,"occasion":"date","time_of_day":"night","weather_note":None,"preferred_color":None,"preferred_style":"smart","conversational_response":None})}]},
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "Edgy outfit for clubbing with friends downtown tonight"}, {"role": "assistant", "content": json.dumps({"intent":"outfit_recommendation","season":None,"occasion":"party","time_of_day":"night","weather_note":None,"preferred_color":None,"preferred_style":"streetwear","conversational_response":None})}]},
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "Can I wear white sneakers with navy chinos to a semi-formal dinner?"}, {"role": "assistant", "content": json.dumps({"intent":"fashion_advice","season":None,"occasion":None,"time_of_day":None,"weather_note":None,"preferred_color":None,"preferred_style":None,"conversational_response":"For styling questions, pairing neutral tones with one statement color is always a safe bet. To get specific outfit recommendations from your wardrobe, tell me what occasion you are dressing for!"})}]},
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "what should i wear?"}, {"role": "assistant", "content": json.dumps({"intent":"clarification_needed","season":None,"occasion":None,"time_of_day":None,"weather_note":None,"preferred_color":None,"preferred_style":None,"conversational_response":"I'd love to help you style an outfit! What event or occasion are you heading to, and what's the vibe you're looking for?"})}]},
    {"messages": [{"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."}, {"role": "user", "content": "I need an outfit for an outdoor summer barbecue"}, {"role": "assistant", "content": json.dumps({"intent":"outfit_recommendation","season":"summer","occasion":"casual","time_of_day":None,"weather_note":"outdoor","preferred_color":None,"preferred_style":"casual","conversational_response":None})}]}
]

# If dataset file exists locally (from repo clone), load full dataset
import os
data_file = 'train_nlu.jsonl'
if os.path.exists(data_file):
    with open(data_file, 'r', encoding='utf-8') as f:
        full_data = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(full_data)} samples from {data_file}")
    dataset_raw = full_data
else:
    print(f"Using {len(sample_data)} bundled training samples")
    dataset_raw = sample_data

dataset = Dataset.from_list(dataset_raw)
print(dataset)

---  
## 🧠 Step 3: Configure LoRA Adapter on Base Model
We load `Qwen/Qwen2.5-1.5B-Instruct` (or `meta-llama/Llama-3.2-1B-Instruct`) and configure low-rank adaptation (LoRA) on `q_proj`, `k_proj`, `v_proj`, and `o_proj`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Alternative: "meta-llama/Llama-3.2-1B-Instruct"

# 4-bit quantization for fast Colab training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    trust_remote_code=True
)

# LoRA Hyperparameters
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

model = get_peft_model(model, peft_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable Parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

---  
## 🚀 Step 4: Fine-Tune with `SFTTrainer` (~3-5 minutes on T4 GPU)
We format the dataset using ChatML and train for 3 epochs with AdamW.

In [ ]:
from trl import SFTTrainer, SFTConfig

# Format chats with tokenizer
def format_prompts(batch):
    formatted_texts = []
    for msgs in batch["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False)
        formatted_texts.append(text)
    return {"text": formatted_texts}

train_ds = dataset.map(format_prompts, batched=True)

training_args = SFTConfig(
    output_dir="./stylemate-lora-adapter",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    peft_config=peft_config,
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

---  
## 🧪 Step 5: Test Inference on Sample Fashion Queries
Evaluate the model's accuracy on extracting structured criteria and fashion intents.

In [ ]:
test_queries = [
    "I have a meeting tomorrow what can I wear",
    "Going to a friend's wedding reception this Saturday evening",
    "Can I wear sneakers with navy chinos?",
    "First date dinner tonight at a rooftop restaurant"
]

model.eval()
for query in test_queries:
    messages = [
        {"role": "system", "content": "You are StyleMate's expert fashion intent parser. Extract structured occasion, season, style, color, and time_of_day as JSON."},
        {"role": "user", "content": query}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.2, do_sample=False)
    
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n📝 Query: '{query}'")
    print(f"🤖 Model Prediction: {generated_text.strip()}")

---  
## 💾 Step 6: Export Adapter Weights for StyleMate Local Inference
Save the LoRA adapter weights and tokenizer to connect directly to StyleMate's `ai/fine-tuning/local_server.py`.

In [ ]:
output_dir = "./stylemate-lora-adapter"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Saved LoRA adapter weights to: {output_dir}")

# Optional: Zip files for direct download to local machine
!zip -r stylemate-lora-adapter.zip ./stylemate-lora-adapter
print("\nDone! Download stylemate-lora-adapter.zip and unzip into StyleMate's ai/fine-tuning/stylemate-lora-adapter directory.")